In [7]:
import os
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PARQUET_PATH = DATA_DIR / "set_price_window.parquet"
REFRESH = True  # set True to re-query the DB

DB_CONFIG = dict(
    host="localhost",
    port=5433,
    dbname="automana",
    user="automana_admin",
    password=os.environ.get("AUTOMANA_DB_PASSWORD", ""),
)

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

In [8]:
SQL = """
SELECT
    s.set_code,
    s.set_name,
    s.released_at,
    r.rarity_name,
    cv.card_version_id,
    ppd.price_date,
    (ppd.price_date - s.released_at)::int AS days_since_release,
    ppd.list_low_cents
FROM pricing.print_price_daily ppd
JOIN card_catalog.card_version cv       ON cv.card_version_id = ppd.card_version_id
JOIN card_catalog.rarities_ref r        ON r.rarity_id = cv.rarity_id
JOIN card_catalog.sets s                ON s.set_id = cv.set_id
JOIN card_catalog.set_type_list_ref st  ON st.set_type_id = s.set_type_id
WHERE st.set_type = 'expansion'
  AND s.released_at BETWEEN '2021-01-01' AND '2026-04-04'
  AND ppd.price_date >= s.released_at
  AND ppd.price_date <= s.released_at + INTERVAL '150 days'
  AND ppd.finish_id = 1
  AND ppd.list_low_cents IS NOT NULL
  AND ppd.list_low_cents > 0
  AND r.rarity_name IN ('common', 'uncommon', 'rare', 'mythic')
"""

if REFRESH or not PARQUET_PATH.exists():
    print("Querying DB…")
    with get_conn() as conn:
        raw = pd.read_sql(SQL, conn, parse_dates=["released_at", "price_date"])
    raw.to_parquet(PARQUET_PATH, index=False)
    print(f"Saved {len(raw):,} rows → {PARQUET_PATH}")
else:
    raw = pd.read_parquet(PARQUET_PATH)
    print(f"Loaded {len(raw):,} rows from parquet")
    assert len(raw) > 0, "Parquet loaded empty — delete it and re-run with REFRESH=True"

print(raw.head(3))

Querying DB…


/tmp/ipykernel_9073/498127294.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  raw = pd.read_sql(SQL, conn, parse_dates=["released_at", "price_date"])


Saved 1,288,627 rows → data/set_price_window.parquet
  set_code                     set_name released_at rarity_name  \
0      stx  Strixhaven: School of Mages  2021-04-23      common   
1      stx  Strixhaven: School of Mages  2021-04-23      common   
2      stx  Strixhaven: School of Mages  2021-04-23      common   

                        card_version_id price_date  days_since_release  \
0  0080bb4a-0f4a-4006-a9ce-f2b2f98c11ac 2021-05-12                  19   
1  0080bb4a-0f4a-4006-a9ce-f2b2f98c11ac 2021-05-11                  18   
2  0080bb4a-0f4a-4006-a9ce-f2b2f98c11ac 2021-05-10                  17   

   list_low_cents  
0               5  
1               5  
2               5  


In [19]:
# Per-card mean price over the full 150-day window
card_avg = (
    raw.groupby(["set_code", "set_name", "rarity_name", "card_version_id"])
    ["list_low_cents"]
    .mean()
    .reset_index(name="avg_cents")
)
print(f"Cards: {len(card_avg):,}")
print(card_avg.head())


Cards: 8,904
  set_code                            set_name rarity_name  \
0      afr  Adventures in the Forgotten Realms      common   
1      afr  Adventures in the Forgotten Realms      common   
2      afr  Adventures in the Forgotten Realms      common   
3      afr  Adventures in the Forgotten Realms      common   
4      afr  Adventures in the Forgotten Realms      common   

                        card_version_id  avg_cents  
0  01dd920d-ff09-4f42-9c39-de7c4066d385   1.039735  
1  02ad2270-4ab9-446e-95fa-d1c081c7de71   1.463576  
2  057f21d6-5c28-4f6d-9ecd-7d6d7b2820fc   4.304636  
3  09d6c31e-4fd0-409c-b1fa-a98d98722529   2.701987  
4  0a2360a4-8b80-4f9a-9dbe-4f10d050918f   1.165563  


In [20]:
def lorenz_and_gini(values):
    v = np.sort(values)
    cumsum = np.cumsum(v)
    lorenz_y = np.concatenate([[0], cumsum / cumsum[-1]])
    lorenz_x = np.linspace(0, 1, len(v) + 1)
    gini = 1 - 2 * np.trapezoid(lorenz_y, lorenz_x)
    return lorenz_x, lorenz_y, gini

pareto_records = []
for set_code, g in card_avg.groupby("set_code", sort=False):
    values = g["avg_cents"].values
    if len(values) < 5:
        continue
    set_name = g["set_name"].iloc[0]
    lx, ly, gini = lorenz_and_gini(values)

    # Sort cards descending by value (most expensive first)
    g_sorted = g.sort_values("avg_cents", ascending=False).reset_index(drop=True)
    g_sorted["cum_value_pct"] = g_sorted["avg_cents"].cumsum() / g_sorted["avg_cents"].sum()
    g_sorted["card_rank_pct"] = (g_sorted.index + 1) / len(g_sorted)

    for threshold in [0.5, 0.8, 0.9]:
        idx = (g_sorted["cum_value_pct"] >= threshold).idxmax()
        pareto_records.append({
            "set_code": set_code,
            "set_name": set_name,
            "threshold": threshold,
            "n_cards_total": len(g_sorted),
            "n_cards_needed": idx + 1,
            "pct_cards_needed": (idx + 1) / len(g_sorted) * 100,
            "gini": gini,
            # Rarity breakdown of the top cards that cover threshold
            **{f"pct_{r}": (g_sorted.loc[:idx, "rarity_name"] == r).sum() / (idx + 1) * 100
               for r in ["mythic", "rare", "uncommon", "common"]},
        })

pareto = pd.DataFrame(pareto_records)
print(pareto[pareto["threshold"] == 0.8][
    ["set_code", "n_cards_total", "n_cards_needed", "pct_cards_needed",
     "pct_mythic", "pct_rare", "pct_uncommon", "pct_common"]
].sort_values("pct_cards_needed").to_string(index=False))


AttributeError: module 'numpy' has no attribute 'trapz'

In [ ]:
x_grid = np.linspace(0, 1, 101)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: one Lorenz curve per set, averaged
ax = axes[0]
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.4, label="Perfect equality")
lorenz_ys = []
for set_code, g in card_avg.groupby("set_code"):
    values = g["avg_cents"].values
    if len(values) < 5:
        continue
    lx, ly, _ = lorenz_and_gini(values)
    lorenz_ys.append(np.interp(x_grid, lx, ly))

stack = np.array(lorenz_ys)
ax.plot(x_grid, np.median(stack, axis=0), color="#4e79a7", linewidth=2.5, label="Median across sets")
ax.fill_between(x_grid,
                np.percentile(stack, 25, axis=0),
                np.percentile(stack, 75, axis=0),
                color="#4e79a7", alpha=0.15, label="IQR")
ax.set_xlabel("Cumulative % of cards (cheapest → most expensive)")
ax.set_ylabel("Cumulative % of total set value")
ax.set_title("Lorenz curve — all cards per set\n(median + IQR across expansion sets 2021–2026)")
ax.legend()

# Right: Gini per set bar chart
ax = axes[1]
gini_df = pareto.drop_duplicates("set_code").sort_values("gini")
ax.barh(gini_df["set_code"], gini_df["gini"], color="#4e79a7", alpha=0.75)
ax.axvline(gini_df["gini"].median(), color="black", linestyle="--", linewidth=1.2,
           label=f"Median = {gini_df['gini'].median():.2f}")
ax.set_xlabel("Gini coefficient  (0 = equal,  1 = one card holds all)")
ax.set_title("Value concentration per set")
ax.legend()

fig.suptitle("Pareto / Lorenz analysis — full set value concentration (nonfoil)", fontsize=13)
plt.tight_layout()
plt.savefig(DATA_DIR / "fig4_pareto_lorenz.png", dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Top row: distribution of % cards needed for 50/80/90% of value
for j, threshold in enumerate([0.5, 0.8, 0.9]):
    ax = axes[0, j]
    sub = pareto[pareto["threshold"] == threshold]["pct_cards_needed"]
    ax.hist(sub, bins=10, color="#4e79a7", edgecolor="white", alpha=0.8)
    ax.axvline(sub.median(), color="black", linestyle="--", linewidth=1.5,
               label=f"Median = {sub.median():.0f}%")
    ax.set_xlabel("% of cards needed")
    ax.set_ylabel("Number of sets")
    ax.set_title(f"{int(threshold * 100)}% of set value")
    ax.set_xlim(0, 100)
    ax.legend()

# Bottom row: rarity composition of top-N cards (at 80% threshold)
sub80 = pareto[pareto["threshold"] == 0.8].sort_values("set_code")
rarity_cols = ["pct_mythic", "pct_rare", "pct_uncommon", "pct_common"]
rarity_labels = ["Mythic", "Rare", "Uncommon", "Common"]
rarity_colors_list = [RARITY_COLORS[r] for r in ["mythic", "rare", "uncommon", "common"]]

ax = axes[1, 0]
x = np.arange(len(sub80))
bottom = np.zeros(len(sub80))
for col, label, color in zip(rarity_cols, rarity_labels, rarity_colors_list):
    ax.bar(x, sub80[col], bottom=bottom, label=label, color=color, alpha=0.85)
    bottom += sub80[col].values
ax.set_xticks(x)
ax.set_xticklabels(sub80["set_code"], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("% of top cards")
ax.set_title("Rarity mix of cards covering 80% of set value")
ax.legend(loc="upper right", fontsize=8)
ax.set_ylim(0, 100)

# Bottom middle: average rarity breakdown across sets
ax = axes[1, 1]
avg_pcts = [sub80[col].mean() for col in rarity_cols]
ax.bar(rarity_labels, avg_pcts, color=rarity_colors_list, alpha=0.85, edgecolor="white")
ax.set_ylabel("Avg % of top cards (across sets)")
ax.set_title("Average rarity composition\nof cards covering 80% of value")
for i, v in enumerate(avg_pcts):
    ax.text(i, v + 0.5, f"{v:.0f}%", ha="center", fontsize=9)

# Bottom right: scatter pct_cards_needed vs gini
ax = axes[1, 2]
sub80_gini = pareto[pareto["threshold"] == 0.8]
ax.scatter(sub80_gini["gini"], sub80_gini["pct_cards_needed"],
           color="#4e79a7", alpha=0.7, edgecolors="white", linewidths=0.5)
for _, row in sub80_gini.iterrows():
    ax.annotate(row["set_code"], (row["gini"], row["pct_cards_needed"]),
                fontsize=6, alpha=0.6, xytext=(2, 2), textcoords="offset points")
ax.set_xlabel("Gini coefficient")
ax.set_ylabel("% of cards for 80% of value")
ax.set_title("Concentration vs Pareto threshold")

fig.suptitle("Pareto analysis — set value concentration (expansion sets 2021–2026, nonfoil)", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_DIR / "fig5_pareto_distribution.png", dpi=150)
plt.show()


In [ ]:
summary_pareto = (
    pareto[pareto["threshold"] == 0.8]
    .agg(
        sets=("set_code", "count"),
        median_pct_cards=("pct_cards_needed", "median"),
        p25=("pct_cards_needed", lambda x: x.quantile(0.25)),
        p75=("pct_cards_needed", lambda x: x.quantile(0.75)),
        avg_pct_mythic=("pct_mythic", "mean"),
        avg_pct_rare=("pct_rare", "mean"),
        avg_pct_uncommon=("pct_uncommon", "mean"),
        avg_pct_common=("pct_common", "mean"),
        median_gini=("gini", "median"),
    )
    .round(1)
    .to_frame("value")
)
display(summary_pareto)

p80 = pareto[pareto["threshold"] == 0.8]
print(f"\nAcross {p80['set_code'].nunique()} expansion sets (2021–2026):")
print(f"  {p80['pct_cards_needed'].median():.0f}% of cards (by count) hold 80% of total set value")
print(f"  IQR: {p80['pct_cards_needed'].quantile(0.25):.0f}% – {p80['pct_cards_needed'].quantile(0.75):.0f}%")
print(f"  Of those top cards: {p80['pct_mythic'].mean():.0f}% mythic, {p80['pct_rare'].mean():.0f}% rare, "
      f"{p80['pct_uncommon'].mean():.0f}% uncommon, {p80['pct_common'].mean():.0f}% common")
print(f"  Median Gini: {p80['gini'].median():.2f}")
